# People — Explore → Extract → Analyse → Ingest

**Graph model produced by this notebook:**

| Node | Key properties |
|------|----------------|
| `People` | id, user_name, user_work_title, user_about, is_active |
| `Country` | name |
| `WorkDivision` | name |
| `Department` | name, division_name |

| Relationship | Pattern |
|---|---|
| `LOCATED_IN` | (People)→(Country) |
| `WORKS_IN_DEPARTMENT` | (People)→(Department) |
| `WORKS_IN_DIVISION` | (People)→(WorkDivision) |
| `HAS_DEPARTMENT` | (WorkDivision)→(Department) |
| `MANAGES` | (People)→(People) — manager→report |

**Sections:**  

0. Setup
1. Explore raw data  
2. Extract & normalise  
3. Analyse extraction results  
4. Ingest into Neo4j  
5. Post-ingestion verification (Cypher)

---
## 0 — Setup

In [1]:
import json
import sys
import pandas as pd
from pathlib import Path
from collections import Counter

# Ensure project root is on sys.path so `src.*` and `config` imports work
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import RAW_FILE, NORMALIZED_DIR, NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD

print("Project root:", PROJECT_ROOT)
print("Raw file:    ", RAW_FILE)
print("Normalized:  ", NORMALIZED_DIR)

Project root: /home/ubuntu/graph_experiments
Raw file:     /home/ubuntu/graph_experiments/data/raw/backyard_dump_19022026_172648.jsonl
Normalized:   /home/ubuntu/graph_experiments/data/normalized


---
## 1 — Explore Raw Data

Understand the shape of the dump before we decide what to extract.

In [2]:
# Load every record from the raw dump
items = []
with open(RAW_FILE, "r") as f:
    for line in f:
        items.append(json.loads(line))

print(f"Total items loaded: {len(items)}")

Total items loaded: 32293


In [3]:
# Object-type distribution
object_type_counter = Counter(item.get("object_type") for item in items)
object_type_counter

Counter({'file': 7877,
         'site_role': 7783,
         'feed': 6029,
         'file_raptor': 4320,
         'content': 3142,
         'feed_comment': 1271,
         'people': 954,
         'native_video': 494,
         'tile': 259,
         'campaign': 97,
         'site': 45,
         'question': 21,
         'app_config': 1})

In [4]:
# Filter to people
people_items = [item for item in items if item.get("object_type") == "people"]
print(f"People items: {len(people_items)}")

People items: 954


In [5]:
people_raw_df = pd.DataFrame(people_items)
people_raw_df.head(3)

,user_slack_connected_as,user_work_title,user_state_country,user_people_category_id,custom_profile_field_24,custom_profile_field_23,custom_profile_field_22,user_type,custom_profile_field_21,user_manager_filter_option,...,custom_profile_field_17,custom_profile_field_16,custom_profile_field_15,custom_profile_field_14,user_street,user_pdr_r_bits,user_pdr_l_bits,user_mobile_phone_numeric,user_derived_field,user_expertises
0,Alejandro Escudero,SMB Account Executive,ON@@CA,NaN,NaN,NaN,NaN,NaN,NaN,Hilary Somers@@0fca0e32-c1ca-4b9b-8f28-7861e65...,...,NaN,NaN,NaN,NaN,NaN,[],[],NaN,NaN,NaN
1,NaN,Contractor,@@,NaN,NaN,NaN,NaN,NaN,NaN,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Rishu Bhardwaj@@ffde564f-ce6c-4f73-a6a0-c9689b...,NaN
2,Pratik Gaikwad,Security Operations Engineer,Maharashatra@@IN,NaN,NaN,NaN,NaN,NaN,NaN,Piyush Rajput@@4e9a7b74-a170-4cd3-8842-21165d5...,...,NaN,NaN,NaN,NaN,NaN,[],[],NaN,Pratik Gaikwad@@21738ad9-630b-4c5a-ba7e-4ffeca...,"[{'name': 'Pentest', 'count': 1, 'id': '4c67d3..."


### 1.1 — Active vs Inactive (raw)

In [6]:
print(people_raw_df["is_active"].value_counts(dropna=False))
print(f"\nActive ratio: {round(people_raw_df['is_active'].mean() * 100, 2)} %")

is_active
True     502
False    452
Name: count, dtype: int64

Active ratio: 52.62 %


### 1.2 — Division / Department null analysis

This is the key analysis that drives our extraction strategy.

In [7]:
# Create boolean columns for the four cases
people_raw_df["has_dept"] = people_raw_df["user_department"].notna() & (people_raw_df["user_department"] != "")
people_raw_df["has_div"]  = people_raw_df["user_work_division"].notna() & (people_raw_df["user_work_division"] != "")

combo = people_raw_df.groupby(["has_div", "has_dept"]).size().reset_index(name="count")
combo["description"] = combo.apply(
    lambda r: {
        (True,  True):  "Both present (normal)",
        (True,  False): "Division only, no department",
        (False, True):  "Department only, no division → will assign _UNASSIGNED_",
        (False, False): "Both null → no org edges",
    }[(r["has_div"], r["has_dept"])],
    axis=1,
)
combo

,has_div,has_dept,count,description
0,False,False,87,Both null → no org edges
1,False,True,2,"Department only, no division → will assign _UN..."
2,True,False,92,"Division only, no department"
3,True,True,773,Both present (normal)


In [8]:
# People with department but no division (the ones we'll assign _UNASSIGNED_)
dept_no_div = people_raw_df[people_raw_df["has_dept"] & ~people_raw_df["has_div"]]
print(f"People with dept but no division: {len(dept_no_div)}")
if len(dept_no_div) > 0:
    print("\nSample departments affected:")
    print(dept_no_div["user_department"].value_counts().head(10))

People with dept but no division: 2

Sample departments affected:
user_department
Engineering              1
Business Applications    1
Name: count, dtype: int64


### 1.3 — Country distribution (raw)

In [9]:
people_raw_df["user_country"].value_counts(dropna=False)

user_country
IN                410
NaN               251
US                227
CA                 30
GB                 23
United States       6
India               5
United Kingdom      1
Canada              1
Name: count, dtype: int64

### 1.4 — Manager coverage (raw)

In [10]:
has_manager = people_raw_df["user_manager_id"].notna().sum()
total = len(people_raw_df)
print(f"People with a manager: {has_manager}/{total} ({round(has_manager/total*100, 1)}%)")

manager_ids = set(people_raw_df["user_manager_id"].dropna())
person_ids  = set(people_raw_df["id"])
missing_managers = manager_ids - person_ids
print(f"Managers referenced but not in dataset: {len(missing_managers)}")

People with a manager: 744/954 (78.0%)
Managers referenced but not in dataset: 0


---
## 2 — Extract & Normalise

Call the extractor from `src/`. Same function used by the headless script.

In [11]:
from src.extractors.people_extractor import extract_people, write_normalized

result = extract_people(RAW_FILE)

print(f"People:         {len(result['people'])}")
print(f"Countries:      {len(result['countries'])}")
print(f"Work divisions: {len(result['work_divisions'])}")
print(f"Departments:    {len(result['departments'])}")

People:         954
Countries:      4
Work divisions: 16
Departments:    60


In [12]:
# Persist normalised files
write_normalized(result, NORMALIZED_DIR)
print(f"\nNormalised files written to: {NORMALIZED_DIR}")


Normalised files written to: /home/ubuntu/graph_experiments/data/normalized


---
## 3 — Analyse Extraction Results

Load the written JSONL files back and verify everything looks correct.

In [13]:
def load_jsonl_df(path):
    """Read a JSONL file into a DataFrame."""
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)


people_df         = load_jsonl_df(NORMALIZED_DIR / "people.jsonl")
countries_df      = load_jsonl_df(NORMALIZED_DIR / "countries.jsonl")
work_divisions_df = load_jsonl_df(NORMALIZED_DIR / "work_divisions.jsonl")
departments_df    = load_jsonl_df(NORMALIZED_DIR / "departments.jsonl")

print(f"People:         {len(people_df)}")
print(f"Countries:      {len(countries_df)}")
print(f"Work Divisions: {len(work_divisions_df)}")
print(f"Departments:    {len(departments_df)}")

People:         954
Countries:      4
Work Divisions: 16
Departments:    60


### 3.1 — People overview

In [14]:
people_df.head()

,id,user_name,user_work_title,user_about,is_active,user_department,user_manager_id,user_work_division,user_country,user_normalized_country
0,ff9f7c59-e0b3-45df-b657-d05aa6f30cae,Alejandro Escudero,SMB Account Executive,NaN,True,Sales & Field Operations,0fca0e32-c1ca-4b9b-8f28-7861e654d833,Sales,CA,Canada
1,ffde564f-ce6c-4f73-a6a0-c9689b5d7265,Rishu Bhardwaj,Contractor,NaN,False,NaN,NaN,Engineering,NaN,NaN
2,21738ad9-630b-4c5a-ba7e-4ffeca6549c3,Pratik Gaikwad,Security Operations Engineer,NaN,True,Security,4e9a7b74-a170-4cd3-8842-21165d52abaf,Engineering,IN,India
3,2b193af5-90a8-47e6-9d1b-1246d533cb8a,Socrates Crawler,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN
4,2b25dc0a-77f9-49fa-9420-85a05198cf43,Himanshu Rana,Senior Software Engineer,NaN,True,Platform Services,3c7ee7e3-e35c-44ad-be99-de8c983d0a08,Engineering,IN,India


In [15]:
# Missing fields after normalisation
missing_summary = people_df.isnull().sum().sort_values(ascending=False)
missing_summary

user_about                 704
user_normalized_country    251
user_country               251
user_manager_id            210
user_department            179
user_work_title             88
user_work_division          87
id                           0
user_name                    0
is_active                    0
dtype: int64

In [16]:
# Active / inactive after normalisation
people_df["is_active"].value_counts(dropna=False)

is_active
True     502
False    452
Name: count, dtype: int64

### 3.2 — Countries (normalised)

In [17]:
print("Unique countries:", countries_df["name"].tolist())
print()
people_df["user_normalized_country"].value_counts(dropna=False)

Unique countries: ['Canada', 'India', 'United Kingdom', 'United States']



user_normalized_country
India             415
NaN               251
United States     233
Canada             31
United Kingdom     24
Name: count, dtype: int64

### 3.3 — Divisions & Departments

In [18]:
print("Work divisions:")
print(work_divisions_df["name"].tolist())

Work divisions:
['AWS Migration', 'Alliances', 'Corporate', 'Customer Experience', 'Engineering', 'Finance & Operations', 'IT', 'Internal Communications', 'Legal', 'Marketing', 'People Operations', 'Product Management', 'Product Ops', 'Rev Ops', 'Sales', '_UNASSIGNED_']


In [19]:
# Departments per division
departments_per_division = (
    departments_df
    .groupby("division_name")
    .size()
    .sort_values(ascending=False)
)
departments_per_division

division_name
Engineering                19
Sales                       7
Marketing                   7
Customer Experience         5
Product Management          4
Finance & Operations        4
People Operations           3
_UNASSIGNED_                2
Alliances                   2
IT                          2
Corporate                   1
AWS Migration               1
Internal Communications     1
Rev Ops                     1
Product Ops                 1
dtype: int64

In [20]:
# Same department name under multiple divisions
dept_conflicts = (
    departments_df
    .groupby("name")["division_name"]
    .nunique()
    .sort_values(ascending=False)
)
conflicts = dept_conflicts[dept_conflicts > 1]
print(f"Department names appearing in multiple divisions: {len(conflicts)}")
conflicts

Department names appearing in multiple divisions: 4


name
Management               5
Business Applications    2
Engineering              2
AWS Migration            2
Name: division_name, dtype: int64

In [21]:
# Show an example conflict if any exist
if len(conflicts) > 0:
    example = conflicts.index[0]
    print(f"Example: '{example}' appears in these divisions:")
    display(departments_df[departments_df["name"] == example])

Example: 'Management' appears in these divisions:


,name,division_name
26,Management,Alliances
27,Management,Customer Experience
28,Management,Finance & Operations
29,Management,Product Management
30,Management,Sales


### 3.4 — People per department

In [22]:
people_per_dept = (
    people_df
    .groupby(["user_department", "user_work_division"])
    .size()
    .sort_values(ascending=False)
)
people_per_dept.head(15)

user_department           user_work_division 
Site and Content          Engineering            61
Sales & Field Operations  Sales                  46
Integrations              Engineering            45
Platform Services         Engineering            39
Professional Services     Customer Experience    39
Product Managers/Owners   Product Management     35
Recognition               Engineering            32
Customer Success          Customer Experience    32
Search and Notifications  Engineering            31
User Experience           Product Management     29
Technical Support         Customer Experience    27
Data Engineering          Engineering            26
Sales Development         Sales                  24
Demand Generation         Marketing              24
Architecture              Engineering            23
dtype: int64

### 3.5 — Manager graph coverage (normalised)

In [23]:
people_df["has_manager"] = people_df["user_manager_id"].notnull()
print(people_df["has_manager"].value_counts())

manager_ids = set(people_df["user_manager_id"].dropna())
person_ids  = set(people_df["id"])
missing_managers = manager_ids - person_ids
print(f"\nManagers referenced but not in dataset: {len(missing_managers)}")
print("(These MANAGES edges will silently not be created — both sides must exist)")

has_manager
True     744
False    210
Name: count, dtype: int64

Managers referenced but not in dataset: 0
(These MANAGES edges will silently not be created — both sides must exist)


### 3.6 — Extraction summary

Quick sanity-check table before we push to Neo4j.

In [24]:
summary = pd.DataFrame([
    {"Entity": "People",       "Count": len(people_df)},
    {"Entity": "Country",      "Count": len(countries_df)},
    {"Entity": "WorkDivision", "Count": len(work_divisions_df)},
    {"Entity": "Department",   "Count": len(departments_df)},
    {"Entity": "--- Edges (expected) ---", "Count": ""},
    {"Entity": "LOCATED_IN",          "Count": people_df["user_normalized_country"].notna().sum()},
    {"Entity": "WORKS_IN_DEPARTMENT", "Count": people_df["user_department"].notna().sum()},
    {"Entity": "WORKS_IN_DIVISION",   "Count": people_df["user_work_division"].notna().sum()},
    {"Entity": "HAS_DEPARTMENT",      "Count": len(departments_df)},
    {"Entity": "MANAGES",             "Count": people_df["user_manager_id"].notna().sum()},
    {"Entity": "  (will fail — Unknown Manager)", "Count": len(missing_managers)},
])
summary

,Entity,Count
0,People,954
1,Country,4
2,WorkDivision,16
3,Department,60
4,--- Edges (expected) ---,
5,LOCATED_IN,703
6,WORKS_IN_DEPARTMENT,775
7,WORKS_IN_DIVISION,867
8,HAS_DEPARTMENT,60
9,MANAGES,744


---
## 4 — Ingest into Neo4j

Import the loader from `src/` and push everything into the graph.

In [25]:
from src.utils.neo4j_client import Neo4jClient
from src.loaders.people_loader import load_people_graph

client = Neo4jClient(
    uri=NEO4J_URI,
    user=NEO4J_USER,
    password=NEO4J_PASSWORD,
)

# Quick connectivity check
print(client.query("RETURN 1 AS ok"))

[{'ok': 1}]


In [26]:
load_people_graph(client, NORMALIZED_DIR, batch=True)

✅ Constraints ensured.


✅ Countries loaded: 4
✅ Work divisions loaded: 16
✅ Departments loaded: 60
✅ People loaded: 954
✅ LOCATED_IN relationships: 703
✅ WORKS_IN_DEPARTMENT relationships: 775
✅ WORKS_IN_DIVISION relationships: 867
✅ MANAGES relationships: 744

✅ People graph fully loaded.


---
## 5 — Post-Ingestion Verification (Cypher)

Query Neo4j to confirm the graph looks right.

In [27]:
# Node counts
node_counts = client.query("""
    CALL {
        MATCH (p:People)       RETURN 'People'       AS label, count(p) AS count
        UNION ALL
        MATCH (c:Country)      RETURN 'Country'      AS label, count(c) AS count
        UNION ALL
        MATCH (w:WorkDivision) RETURN 'WorkDivision' AS label, count(w) AS count
        UNION ALL
        MATCH (d:Department)   RETURN 'Department'   AS label, count(d) AS count
    }
    RETURN label, count
""")
pd.DataFrame(node_counts)

,label,count
0,People,954
1,Country,4
2,WorkDivision,16
3,Department,60


In [28]:
# Relationship counts
rel_counts = client.query("""
    CALL {
        MATCH ()-[r:LOCATED_IN]->()          RETURN 'LOCATED_IN'          AS type, count(r) AS count
        UNION ALL
        MATCH ()-[r:WORKS_IN_DEPARTMENT]->() RETURN 'WORKS_IN_DEPARTMENT' AS type, count(r) AS count
        UNION ALL
        MATCH ()-[r:WORKS_IN_DIVISION]->()   RETURN 'WORKS_IN_DIVISION'   AS type, count(r) AS count
        UNION ALL
        MATCH ()-[r:HAS_DEPARTMENT]->()      RETURN 'HAS_DEPARTMENT'      AS type, count(r) AS count
        UNION ALL
        MATCH ()-[r:MANAGES]->()             RETURN 'MANAGES'             AS type, count(r) AS count
    }
    RETURN type, count
""")
pd.DataFrame(rel_counts)

,type,count
0,LOCATED_IN,703
1,WORKS_IN_DEPARTMENT,775
2,WORKS_IN_DIVISION,867
3,HAS_DEPARTMENT,60
4,MANAGES,744


In [29]:
# Sample: people per country
country_dist = client.query("""
    MATCH (p:People)-[:LOCATED_IN]->(c:Country)
    RETURN c.name AS country, count(p) AS people
    ORDER BY people DESC
""")
pd.DataFrame(country_dist)

,country,people
0,India,415
1,United States,233
2,Canada,31
3,United Kingdom,24


In [30]:
# Sample: top 10 departments by headcount
top_depts = client.query("""
    MATCH (p:People)-[:WORKS_IN_DEPARTMENT]->(d:Department)<-[:HAS_DEPARTMENT]-(w:WorkDivision)
    RETURN w.name AS division, d.name AS department, count(p) AS people
    ORDER BY people DESC
    LIMIT 10
""")
pd.DataFrame(top_depts)

,division,department,people
0,Engineering,Site and Content,61
1,Sales,Sales & Field Operations,46
2,Engineering,Integrations,45
3,Customer Experience,Professional Services,39
4,Engineering,Platform Services,39
5,Product Management,Product Managers/Owners,35
6,Engineering,Recognition,32
7,Customer Experience,Customer Success,32
8,Engineering,Search and Notifications,31
9,Product Management,User Experience,29


In [31]:
# Sample: managers with most direct reports
top_managers = client.query("""
    MATCH (mgr:People)-[:MANAGES]->(p:People)
    RETURN mgr.user_name AS manager, count(p) AS direct_reports
    ORDER BY direct_reports DESC
    LIMIT 10
""")
pd.DataFrame(top_managers)

,manager,direct_reports
0,Anuj Gupta,27
1,Sabika Nazim,24
2,Kim Luu,23
3,Lokesh Kumar,17
4,Harish Rathor,16
5,Pushker Yadav,16
6,Deepak Soni,15
7,Matt Furlong,15
8,Kay Lim,13
9,Patrick Kilcommons,13


In [32]:
# People without any org edges (no dept, no country)
orphans = client.query("""
    MATCH (p:People)
    WHERE NOT (p)-[:WORKS_IN_DEPARTMENT]->() AND NOT (p)-[:LOCATED_IN]->()
    RETURN count(p) AS orphan_people
""")
print(f"People with no WORKS_IN_DEPARTMENT and no LOCATED_IN: {orphans[0]['orphan_people']}")

People with no WORKS_IN_DEPARTMENT and no LOCATED_IN: 178


In [33]:
client.close()
print("Neo4j connection closed.")

Neo4j connection closed.
